In [1]:
import pandas as pd

/var/folders/vt/p3vf9d352l7fsw1fclffgfsh0000gn/T/ipykernel_79494/4080736814.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [19]:
df = pd.read_csv('/Users/ganfeng/Desktop/taxis.csv')

In [20]:
df["pick_up_dt"] = pd.to_datetime(df["pickup"])

In [21]:
df["dropoff_dt"] = pd.to_datetime(df["dropoff"])

In [22]:
df["duration"] = df["dropoff_dt"] - df["pick_up_dt"]

In [23]:
df["duration_num"] = df["duration"].dt.total_seconds()

In [24]:
df["duration_num"]

0        375.0
1        425.0
2        444.0
3       1552.0
4        572.0
         ...  
6428     214.0
6429    3383.0
6430    1147.0
6431     304.0
6432    1000.0
Name: duration_num, Length: 6433, dtype: float64

In [27]:
df = df.drop(['pickup', 'dropoff', 'pick_up_dt', 'dropoff_dt', 'duration'], axis=1)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   passengers       6433 non-null   int64  
 1   distance         6433 non-null   float64
 2   fare             6433 non-null   float64
 3   tip              6433 non-null   float64
 4   tolls            6433 non-null   float64
 5   total            6433 non-null   float64
 6   color            6433 non-null   object 
 7   payment          6389 non-null   object 
 8   pickup_zone      6407 non-null   object 
 9   dropoff_zone     6388 non-null   object 
 10  pickup_borough   6407 non-null   object 
 11  dropoff_borough  6388 non-null   object 
 12  duration_num     6433 non-null   float64
dtypes: float64(6), int64(1), object(6)
memory usage: 653.5+ KB


In [30]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

target = 'total'
X = df.drop(target, axis=1)

In [42]:
y = df[target]

In [31]:
X.dtypes

passengers           int64
distance           float64
fare               float64
tip                float64
tolls              float64
color               object
payment             object
pickup_zone         object
dropoff_zone        object
pickup_borough      object
dropoff_borough     object
duration_num       float64
dtype: object

In [34]:
categorial_variable = X.select_dtypes(include=['object']).columns.tolist()

In [35]:
categorial_variable

['color',
 'payment',
 'pickup_zone',
 'dropoff_zone',
 'pickup_borough',
 'dropoff_borough']

In [36]:
numerical_variable = set(X.columns) - set(categorial_variable)

In [37]:
numerical_variable

{'distance', 'duration_num', 'fare', 'passengers', 'tip', 'tolls'}

In [40]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [41]:
X_encoded.head()

,passengers,distance,fare,tip,tolls,duration_num,color_yellow,payment_credit card,pickup_zone_Alphabet City,pickup_zone_Astoria,...,dropoff_zone_World Trade Center,dropoff_zone_Yorkville East,dropoff_zone_Yorkville West,pickup_borough_Brooklyn,pickup_borough_Manhattan,pickup_borough_Queens,dropoff_borough_Brooklyn,dropoff_borough_Manhattan,dropoff_borough_Queens,dropoff_borough_Staten Island
0,1,1.60,7.0,2.15,0.0,375.0,True,True,False,False,...,False,False,False,False,True,False,False,True,False,False
1,1,0.79,5.0,0.00,0.0,425.0,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
2,1,1.37,7.5,2.36,0.0,444.0,True,True,True,False,...,False,False,False,False,True,False,False,True,False,False
3,1,7.70,27.0,6.15,0.0,1552.0,True,True,False,False,...,False,False,True,False,True,False,False,True,False,False
4,3,2.16,9.0,1.10,0.0,572.0,True,True,False,False,...,False,False,True,False,True,False,False,True,False,False


In [51]:
train_x, test_x, train_y, test_y = train_test_split(X_encoded, y, test_size=0.3, random_state=42)

In [52]:
rf = RandomForestRegressor(n_estimators=100, max_depth=2, random_state=42)

In [53]:
rf.fit(train_x, train_y)

RandomForestRegressor(max_depth=2, random_state=42)

In [54]:
test_x['pred'] = rf.predict(test_x)

In [56]:
test_x[test_x['color_yellow'] == True]['pred'].mean()

18.24015142135284

In [57]:
test_x[test_x['color_yellow'] == False]['pred'].mean()

18.944247508939803